# Strands 에이전트와 AgentCore 메모리 (장기 메모리) - 커스텀 전략 오버라이드

## 소개

이 튜토리얼은 **MemoryManager**와 **커스텀 전략** 및 **MemorySessionManager**를 사용하여 훅을 통해 AgentCore 메모리와 통합된 Strands 에이전트를 사용하여 **지능형 고객 지원 에이전트**를 구축하는 방법을 보여줍니다. 고객 상호작용 이력에 대한 장기 메모리, 구매 세부정보 기억, 이전 대화와 사용자 선호도를 기반으로 한 개인화된 지원 제공에 초점을 맞춥니다.

**참고: 이 접근 방식은 커스텀 전략(CustomSemanticStrategy, CustomUserPreferenceStrategy)을 사용하며, IAM 실행 역할이 필요하고 추출 및 통합을 위한 커스텀 모델을 지정할 수 있습니다.**

### 튜토리얼 세부사항

| 항목         | 세부내용                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 장기 대화형                                                         |
| 에이전트 유형          | 고객 지원                                                                 |
| 에이전트 프레임워크   | Strands Agents                                                                   |
| LLM 모델           | Anthropic Claude Haiku 4.5                                                      |
| 튜토리얼 구성요소 | AgentCore 시맨틱 및 사용자 선호도 메모리 추출 (모델 오버라이드를 사용한 커스텀), 메모리 저장 및 검색을 위한 훅              |
| 예제 난이도  | 고급                                                                         |

학습 내용:
- MemoryManager를 사용하여 커스텀 장기 전략으로 AgentCore 메모리 설정
- 추출 및 통합을 위한 커스텀 모델 설정
- 커스텀 전략 모델 호출을 위한 IAM 실행 역할 생성
- MemorySessionManager를 사용한 자동 저장 및 검색 메모리 훅 생성
- 영구 메모리를 갖춘 고객 지원 에이전트 구축
- 이전 상호작용의 컨텍스트로 고객 이슈 처리

### 시나리오 컨텍스트
이 예제에서는 **고객 지원 사용 사례**를 구축합니다. 에이전트는 주문 이력, 선호도, 이전 이슈를 포함한 고객 컨텍스트를 기억하여 더 개인화되고 효과적인 지원을 제공합니다. 고객과의 대화는 메모리 훅을 사용하여 자동으로 저장되어 중요한 세부사항이 절대 손실되지 않습니다. 시맨틱 및 사용자 선호도와 같은 여러 메모리 전략을 사용하여 에이전트는 다양한 관련 정보를 캡처할 수 있습니다. 이 설정을 통해 에이전트는 고객의 이력과 선호도를 완전히 인식한 상태에서 이슈를 해결할 수 있습니다. 또한 에이전트는 웹 검색 기능과 통합되어 있어 최신 제품 정보와 문제 해결 가이드를 쉽게 제공할 수 있습니다.

## 아키텍처

<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>


## 사전 요구사항

이 튜토리얼을 실행하려면 다음이 필요합니다:
- Python 3.10+
- Amazon Bedrock AgentCore 메모리 권한이 있는 AWS 자격 증명
- MemoryManager를 지원하는 Amazon Bedrock AgentCore SDK

## 빌트인 전략 vs 커스텀 전략 선택하기

AgentCore 메모리는 메모리 추출 및 통합을 위한 두 가지 접근 방식을 제공합니다. 이 노트북은 **커스텀 전략 오버라이드** 접근 방식을 시연합니다.

### 커스텀 전략 오버라이드 (이 노트북)

**사용 시기:**
- 추출/통합을 위한 커스텀 Bedrock 모델을 지정해야 할 때
- 모델 동작에 대한 세밀한 제어가 필요할 때
- 추출 및 통합을 위한 커스텀 프롬프트가 필요할 때
- 특정 모델 기능이 필요한 고급 사용 사례
- 특정 모델 버전에 대한 규정 준수 요구사항

**주요 특징:**
- `CustomSemanticStrategy` 및 `CustomUserPreferenceStrategy` 클래스 사용
- 모델 호출을 위한 IAM 실행 역할 필요
- `ExtractionConfig` 및 `ConsolidationConfig` 지정 가능
- 더 복잡한 설정이지만 더 큰 제어력
- 특수한 요구사항에 유용

**예시:**
```python
strategies = [
    CustomSemanticStrategy(
        name="CustomerSupportSemantic",
        extraction_config=ExtractionConfig(
            model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",
            append_to_prompt="Extract factual information..."
        ),
        consolidation_config=ConsolidationConfig(
            model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",
            append_to_prompt="Consolidate semantic insights..."
        ),
        namespaces=["support/customer/{actorId}/semantic/"]
    )
]

memory = memory_manager.get_or_create_memory(
    name="CustomerSupportMemory",
    strategies=strategies,
    memory_execution_role_arn=MEMORY_EXECUTION_ROLE_ARN  # 필수!
)
```

### 빌트인 전략 (`customer-support-inbuilt-strategy.ipynb` 참조)

**사용 시기:**
- 빠른 설정 및 프로토타이핑
- 표준 메모리 추출 요구사항
- 커스텀 모델 선택이 필요 없는 경우
- 간소화된 IAM 설정 (실행 역할 불필요)
- 기본 AgentCore 모델을 사용하는 프로덕션 워크로드

**주요 특징:**
- `SemanticStrategy` 및 `UserPreferenceStrategy` 클래스 사용
- AgentCore 메모리가 모델을 자동 선택 및 관리
- IAM 실행 역할 불필요
- 더 적은 파라미터로 간단한 설정
- 대부분의 사용 사례에 이상적

**예시:**
```python
strategies = [
    SemanticStrategy(
        name="CustomerSupportSemantic",
        description="Stores facts from conversations",
        namespaces=["support/customer/{actorId}/semantic/"]
    )
]

memory = memory_manager.get_or_create_memory(
    name="CustomerSupportMemory",
    strategies=strategies
    # memory_execution_role_arn 불필요!
)
```

### 빠른 비교 표

| 기능 | 빌트인 전략 | 커스텀 전략 오버라이드 |
|---------|---------------------|-------------------------|
| **설정 복잡도** | 간단 | 고급 |
| **IAM 역할 필요** | 아니오 | 예 |
| **모델 선택** | 자동 (AgentCore 관리) | 수동 (직접 지정) |
| **커스텀 프롬프트** | 아니오 | 예 |
| **설정** | 최소 | 상세 |
| **사용 사례** | 표준 메모리 추출 | 커스텀 모델 요구사항 |
| **추천 대상** | 대부분의 애플리케이션 | 특수한 요구사항 |

---

**권장:** 대부분의 사용 사례에는 빌트인 전략(`customer-support-inbuilt-strategy.ipynb`)으로 시작하세요. 특정 모델 요구사항이 있거나 추출/통합 동작에 대한 세밀한 제어가 필요한 경우에만 이 커스텀 전략 오버라이드 접근 방식을 사용하세요.

## 1단계: 의존성 설치 및 설정
필요한 모든 라이브러리를 임포트하고 이 노트북이 작동하도록 클라이언트를 정의합니다.

In [1]:
!pip install -qr requirements.txt

In [ ]:
import logging
import json
from typing import Dict, List
from datetime import datetime
from botocore.exceptions import ClientError

# 로깅 설정
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("customer-support")

# Strands Agent에 필요한 모듈 임포트
from strands import Agent, tool
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent
from ddgs import DDGS

# 메모리 관리 모듈 임포트
from bedrock_agentcore_starter_toolkit.operations.memory.manager import Memory, MemoryManager
from bedrock_agentcore_starter_toolkit.operations.memory.models.strategies import (
    SemanticStrategy, SummaryStrategy, CustomSemanticStrategy, CustomSummaryStrategy, CustomUserPreferenceStrategy,
    ExtractionConfig, ConsolidationConfig
)
from bedrock_agentcore.memory.constants import (
    BlobMessage, ConversationalMessage, MessageRole, RetrievalConfig
)
from bedrock_agentcore.memory.models import (
    StringValue, EventMetadataFilter, LeftExpression, RightExpression, OperatorType, MemoryRecord
)
from bedrock_agentcore.memory.session import Actor, MemorySession, MemorySessionManager

# 메시지 역할 상수 정의
USER = MessageRole.USER
ASSISTANT = MessageRole.ASSISTANT

logger.info("모든 임포트가 성공적으로 로드되었습니다")

In [ ]:
# 설정 - 올바른 값으로 교체하세요
REGION = "us-east-1"  
CUSTOMER_ID = "customer_001"
SESSION_ID = f"support_{datetime.now().strftime('%Y%m%d%H%M%S')}"

# IAM 역할 생성을 위한 boto3 임포트
import boto3
import json as json_module
from botocore.exceptions import ClientError

logger.info(f"설정이 로드되었습니다")
logger.info(f"   리전: {REGION}")
logger.info(f"   고객 ID: {CUSTOMER_ID}")
logger.info(f"   세션 ID: {SESSION_ID}")

## 1.1단계: 커스텀 메모리 전략을 위한 IAM 역할 생성

커스텀 메모리 전략에는 AgentCore 메모리가 추출 및 통합을 위해 Bedrock 모델을 호출할 수 있도록 실행 역할이 필요합니다. 이 역할은 `CustomSemanticStrategy` 또는 `CustomUserPreferenceStrategy`를 사용할 때 필요합니다.

In [ ]:
# AgentCore 메모리 커스텀 전략을 위한 IAM 역할 생성
def create_memory_execution_role():
    """필요한 권한이 있는 AgentCore 메모리 커스텀 전략용 IAM 역할을 생성합니다"""
    iam_client = boto3.client('iam', region_name=REGION)
    
    # 현재 AWS 계정 ID 가져오기
    sts_client = boto3.client('sts', region_name=REGION)
    account_id = sts_client.get_caller_identity()['Account']
    
    role_name = "AgentCoreMemoryExecutionRole"
    role_arn = f"arn:aws:iam::{account_id}:role/{role_name}"
    
    # AgentCore 메모리 서비스를 위한 신뢰 정책
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "",
                "Effect": "Allow",
                "Principal": {
                    "Service": [
                        "bedrock-agentcore.amazonaws.com"
                    ]
                },
                "Action": "sts:AssumeRole",
                "Condition": {
                    "StringEquals": {
                        "aws:SourceAccount": account_id
                    },
                    "ArnLike": {
                        "aws:SourceArn": f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:*"
                    }
                }
            }
        ]
    }
    
    # Bedrock 모델 호출을 위한 권한 정책
    permissions_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "bedrock:InvokeModel",
                    "bedrock:InvokeModelWithResponseStream"
                ],
                "Resource": [
                    "arn:aws:bedrock:*::foundation-model/*",
                    "arn:aws:bedrock:*:*:inference-profile/*"
                ],
                "Condition": {
                    "StringEquals": {
                        "aws:ResourceAccount": account_id
                    }
                }
            }
        ]
    }
    
    try:
        # 역할이 이미 존재하는지 확인합니다
        try:
            existing_role = iam_client.get_role(RoleName=role_name)
            logger.info(f"IAM 역할이 이미 존재합니다: {role_arn}")
            return role_arn
        except ClientError as e:
            if e.response['Error']['Code'] != 'NoSuchEntity':
                raise
        
        # 역할을 생성합니다
        logger.info(f"IAM 역할 생성 중: {role_name}")
        iam_client.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json_module.dumps(trust_policy),
            Description="Execution role for AgentCore Memory custom strategies",
            Tags=[
                {
                    'Key': 'Purpose',
                    'Value': 'AgentCoreMemory'
                }
            ]
        )
        
        # 권한 정책을 연결합니다
        policy_name = "AgentCoreMemoryBedrockAccess"
        iam_client.put_role_policy(
            RoleName=role_name,
            PolicyName=policy_name,
            PolicyDocument=json_module.dumps(permissions_policy)
        )
        
        logger.info(f"IAM 역할이 성공적으로 생성되었습니다: {role_arn}")
        logger.info(f"   - 신뢰 정책: AgentCore 메모리 서비스가 이 역할을 위임할 수 있습니다")
        logger.info(f"   - 권한: bedrock:InvokeModel 및 bedrock:InvokeModelWithResponseStream")
        
        return role_arn
        
    except ClientError as e:
        error_code = e.response['Error']['Code']
        if error_code == 'AccessDenied':
            logger.error("IAM 역할 생성 접근 거부. IAM 권한이 있는지 확인하세요:")
            logger.error("   - iam:CreateRole")
            logger.error("   - iam:PutRolePolicy")
            logger.error("   - iam:GetRole")
        else:
            logger.error(f"IAM 역할 생성 실패: {e}")
        raise
    except Exception as e:
        logger.error(f"IAM 역할 생성 중 예상치 못한 오류: {e}")
        raise

# 실행 역할 생성
try:
    MEMORY_EXECUTION_ROLE_ARN = create_memory_execution_role()
    logger.info(f"메모리 실행 역할 준비 완료: {MEMORY_EXECUTION_ROLE_ARN}")
except Exception as e:
    logger.error(f"메모리 실행 역할 생성 실패: {e}")
    # 데모 목적으로, 생성 실패 시 역할 ARN을 수동으로 설정할 수 있습니다
    # MEMORY_EXECUTION_ROLE_ARN = "arn:aws:iam::YOUR_ACCOUNT_ID:role/YourExistingRole"
    raise

## 2단계: 고객 지원을 위한 메모리 리소스 생성

고객 지원을 위해 여러 메모리 전략을 사용합니다:
- **CustomUserPreferenceStrategy**: 고객 선호도와 행동을 캡처합니다
- **CustomSemanticStrategy**: 주문 사실과 제품 정보를 저장합니다

**중요**: 커스텀 전략에는 AgentCore 메모리가 Bedrock 모델을 호출할 수 있도록 IAM 실행 역할이 필요합니다. 이 역할은 위의 1.1단계에서 생성했습니다.

In [ ]:
# Memory Manager 초기화
memory_manager = MemoryManager(region_name=REGION)
memory_name = "CustomerSupportLongTermMemory"

# 기본 메모리 매니저 초기화 및 연결 테스트
logger.info(f"MemoryManager가 리전 {REGION}에 대해 초기화되었습니다")
logger.info(f"메모리 매니저 타입: {type(memory_manager)}")

# 타입 클래스를 사용하여 메모리 전략 정의
strategies = [
    CustomUserPreferenceStrategy(
        name="CustomerPreferences",
        description="Captures customer preferences and behavior",
        extraction_config=ExtractionConfig(
            append_to_prompt="Extract customer preferences and behavior patterns",
            model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0"
        ),
        consolidation_config=ConsolidationConfig(
            append_to_prompt="Consolidate customer preferences",
            model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0"
        ),
        namespaces=["support/customer/{actorId}/preferences/"]
    ),
    CustomSemanticStrategy(
        name="CustomerSupportSemantic",
        description="Stores facts from conversations",
        extraction_config=ExtractionConfig(
            append_to_prompt="Extract factual information from customer support conversations",
            model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0"
        ),
        consolidation_config=ConsolidationConfig(
            append_to_prompt="Consolidate semantic insights from support interactions",
            model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0"
        ),
        namespaces=["support/customer/{actorId}/semantic/"]
    )
]

# 전략 설정 검증
logger.info(f"{len(strategies)}개의 메모리 전략이 설정되었습니다:")
for i, strategy in enumerate(strategies, 1):
    logger.info(f"  {i}. {strategy.name} ({type(strategy).__name__})")
    logger.info(f"     설명: {strategy.description}")
    logger.info(f"     네임스페이스: {strategy.namespaces}")
    if hasattr(strategy, 'extraction_config') and strategy.extraction_config:
        logger.info(f"     추출 모델: {strategy.extraction_config.model_id}")
    if hasattr(strategy, 'consolidation_config') and strategy.consolidation_config:
        logger.info(f"     통합 모델: {strategy.consolidation_config.model_id}"),

# MemoryManager를 사용하여 메모리 리소스 생성
logger.info(f"'{memory_name}' 메모리를 {len(strategies)}개의 전략으로 생성 중...")

try:
    memory = memory_manager.get_or_create_memory(
        name=memory_name,
        strategies=strategies,         # 타입화된 전략 객체 전달
        description="Memory for customer support agent",
        event_expiry_days=90,          # 90일 후 메모리 만료
        memory_execution_role_arn=MEMORY_EXECUTION_ROLE_ARN,  # 커스텀 전략에 필수
    )
    memory_id = memory.id
    logger.info(f"MemoryManager를 통해 메모리 생성/검색 성공:")
    logger.info(f"   메모리 ID: {memory_id}")
    logger.info(f"   메모리 이름: {memory.name}")
    logger.info(f"   메모리 상태: {memory.status}")
    
except Exception as e:
    # 메모리 생성 중 오류를 향상된 오류 보고와 함께 처리합니다
    logger.error(f"메모리 생성 실패: {e}")
    logger.error(f"오류 타입: {type(e).__name__}")
    import traceback
    traceback.print_exc()
    
    # 오류 시 정리 - 부분적으로 생성된 메모리를 삭제합니다
    if 'memory_id' in locals():
        try:
            logger.info(f"부분적으로 생성된 메모리 정리 시도 중: {memory_id}")
            memory_manager.delete_memory(memory_id)
            logger.info(f"메모리 정리 성공: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"메모리 정리 실패: {cleanup_error}")
    
    # 원래 예외를 다시 발생시킵니다
    raise

In [ ]:
# 메모리 매니저 기본 기능 테스트
try:
    # 기존 메모리 목록 조회 테스트
    existing_memories = memory_manager.list_memories()
    logger.info(f"메모리 매니저 연결 성공. {len(existing_memories)}개의 기존 메모리를 찾았습니다")
    
    # 기존 테스트 메모리 정리
    for mem in existing_memories:
        if mem.name and mem.name.startswith("CustomerSupportLongTermMemory"):
            logger.info(f"기존 테스트 메모리 정리 중: {mem.id}")
            memory_manager.delete_memory(mem.id)
            
except Exception as e:
    logger.error(f"메모리 매니저 테스트 실패: {e}")
    raise

메모리에 할당한 전략이 포함되어 있는지 확인합니다

In [ ]:
# MemoryManager를 사용하여 메모리 정보 표시
print(f"메모리 ID: {memory.id}")
print(f"메모리 이름: {memory.name}")
print(f"메모리 설명: {memory.description}")
print(f"메모리 상태: {memory.status}")
print(f"전략 수: {len(strategies)}")
for i, strategy in enumerate(strategies, 1):
    print(f"  {i}. {strategy.name}: {strategy.description}")

## 3단계: 에이전트 도구 생성

In [ ]:
from ddgs.exceptions import DDGSException, RatelimitException
from ddgs import DDGS

@tool
def web_search(query: str, max_results: int = 3) -> str:
    """Search the web for product information, troubleshooting guides, or support articles.
    
    Args:
        query: Search query for product info or troubleshooting
        max_results: Maximum number of results to return
    
    Returns:
        Search results with titles and snippets
    """
    try:
        results = DDGS().text(query, region="us-en", max_results=max_results)
        if not results:
            return "No search results found."
        
        formatted_results = []
        for i, result in enumerate(results, 1):
            formatted_results.append(f"{i}. {result.get('title', 'No title')}\n   {result.get('body', 'No description')}")
        
        return "\n".join(formatted_results)
    except RatelimitException:
        return "Rate limit reached: Please try again after a short delay."
    except DDGSException as d:
        return f"Search Error: {d}"
    except Exception as e:
        return f"Search error: {str(e)}"

logger.info("웹 검색 도구 준비 완료")

@tool
def check_order_status(order_number: str) -> str:
    """Check the status of a customer order.
    
    Args:
        order_number: The order number to check
    
    Returns:
        Order status information
    """
    # 주문 조회 시뮬레이션
    mock_orders = {
        "123456": "iPhone 15 Pro - Delivered on June 5, 2025",
        "654321": "Sennheiser Headphones - Delivered on June 25, 2025, 1-year warranty active",
        "789012": "Samsung Galaxy S23 - In transit, expected delivery on July 1, 2025",
    }
    
    return mock_orders.get(order_number, f"Order {order_number} not found. Please verify the order number.")

logger.info("주문 상태 확인 도구 준비 완료")

## 4단계: 세션 매니저 초기화

**새로운 기능: 이 섹션에서는 세션 기반 메모리 작업을 위한 MemorySessionManager를 소개합니다.**

In [ ]:
# 세션 메모리 매니저 초기화
session_manager: MemorySessionManager = MemorySessionManager(memory_id=memory.id, region_name=REGION)

# 특정 고객을 위한 메모리 세션 생성
customer_session: MemorySession = session_manager.create_memory_session(
    actor_id=CUSTOMER_ID, 
    session_id=SESSION_ID
)

logger.info(f"메모리 {memory.id}에 대한 세션 매니저가 초기화되었습니다")
logger.info(f"actor {CUSTOMER_ID}에 대한 고객 세션이 생성되었습니다")
logger.info(f"   세션 타입: {type(customer_session)}")
logger.info(f"   Actor 객체: {customer_session.get_actor()}")

## 5단계: 고객 지원을 위한 메모리 훅 프로바이더 생성
훅은 에이전트 실행 라이프사이클의 특정 시점에서 실행되는 특수 함수입니다. 커스텀 훅 프로바이더는 다음을 통해 고객 지원 컨텍스트를 자동으로 관리합니다:
- 세션 기반 메서드를 사용하여 각 응답 후 **지원 상호작용을 저장**합니다
- 새 쿼리를 처리할 때 이전 주문과 선호도에서 **관련 컨텍스트를 검색하고 주입**합니다.

In [ ]:
class CustomerSupportMemoryHooks(HookProvider):
    """고객 지원 에이전트를 위한 메모리 훅 - MemorySession으로 강화"""
    
    def __init__(self, customer_session: MemorySession):
        # MemorySession을 직접 받습니다
        self.customer_session = customer_session
        
        # 다양한 메모리 유형에 대한 검색 설정 정의
        self.retrieval_config = {
            "support/customer/{actorId}/preferences/": RetrievalConfig(top_k=3, relevance_score=0.3),
            "support/customer/{actorId}/semantic/": RetrievalConfig(top_k=5, relevance_score=0.2)
        }
    
    def retrieve_customer_context(self, event: MessageAddedEvent):
        """MemorySession을 사용하여 지원 쿼리 처리 전 고객 컨텍스트를 검색합니다"""
        messages = event.agent.messages
        if messages[-1]["role"] == "user" and "toolResult" not in messages[-1]["content"][0]:
            user_query = messages[-1]["content"][0]["text"]
            
            try:
                # 컨텍스트 검색을 위해 MemorySession 사용
                relevant_memories = []
                
                # MemorySession을 사용하여 다양한 메모리 네임스페이스를 검색합니다
                for namespace_template, config in self.retrieval_config.items():
                    # 세션의 실제 actor ID로 네임스페이스 템플릿을 해결합니다
                    resolved_namespace = namespace_template.format(
                        actorId=self.customer_session._actor_id
                    )
                    
                    # MemorySession API 사용 (actor_id/session_id를 전달할 필요 없음)
                    memories = self.customer_session.search_long_term_memories(
                        query=user_query,
                        namespace_prefix=resolved_namespace,
                        top_k=config.top_k
                    )
                    
                    # 관련성 점수로 필터링합니다
                    filtered_memories = [
                        memory for memory in memories
                        if memory.get("score", 0) >= config.relevance_score
                    ]
                    
                    relevant_memories.extend(filtered_memories)
                    logger.info(f"{resolved_namespace}에서 {len(filtered_memories)}개의 관련 메모리를 찾았습니다 (전체 {len(memories)}개에서 필터링)")
                
                # 메모리가 있으면 에이전트의 시스템 프롬프트에 컨텍스트를 주입합니다
                if relevant_memories:
                    context_text = self._format_context(relevant_memories)
                    original_prompt = event.agent.system_prompt
                    enhanced_prompt = f"{original_prompt}\n\nCustomer Context:\n{context_text}"
                    event.agent.system_prompt = enhanced_prompt
                    logger.info(f"{len(relevant_memories)}개의 메모리가 에이전트 컨텍스트에 주입되었습니다")
                    
            except Exception as e:
                logger.error(f"고객 컨텍스트 검색 실패: {e}")
    
    def _format_context(self, memories: List[MemoryRecord]) -> str:
        """검색된 메모리를 에이전트 컨텍스트에 맞게 포맷합니다"""
        context_lines = []
        for i, memory in enumerate(memories[:5], 1):  # 상위 5개로 제한
            content = memory.get('content', {}).get('text', 'No content available')
            score = memory.get('score', 0)
            context_lines.append(f"{i}. (점수: {score:.2f}) {content[:200]}...")
        
        return "\n".join(context_lines)
    
    def save_support_interaction(self, event: AfterInvocationEvent):
        """MemorySession을 사용하여 지원 상호작용을 저장합니다 (더 깔끔한 API)"""
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                # 마지막 고객 쿼리와 에이전트 응답을 가져옵니다
                customer_query = None
                agent_response = None
                
                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        agent_response = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not customer_query and "toolResult" not in msg["content"][0]:
                        customer_query = msg["content"][0]["text"]
                        break
                
                if customer_query and agent_response:
                    # MemorySession 사용 (actor_id/session_id를 전달할 필요 없음)
                    interaction_messages = [
                        ConversationalMessage(customer_query, USER),
                        ConversationalMessage(agent_response, ASSISTANT)
                    ]
                    
                    result = self.customer_session.add_turns(interaction_messages)
                    logger.info(f"MemorySession을 사용하여 상호작용 저장 완료 - 이벤트 ID: {result['eventId']}")
                    
        except Exception as e:
            logger.error(f"지원 상호작용 저장 실패: {e}")
    
    def register_hooks(self, registry: HookRegistry) -> None:
        """고객 지원 메모리 훅을 등록합니다"""
        registry.add_callback(MessageAddedEvent, self.retrieve_customer_context)
        registry.add_callback(AfterInvocationEvent, self.save_support_interaction)
        logger.info("MemorySession과 함께 고객 지원 메모리 훅이 등록되었습니다")
print('실행 완료!')

### 6단계: 고객 지원 에이전트 생성

In [ ]:
# MemorySession을 사용하여 메모리 훅 생성
support_hooks = CustomerSupportMemoryHooks(customer_session)

# 고객 지원 에이전트 생성
support_agent = Agent(
    hooks=[support_hooks],
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[web_search, check_order_status],
    system_prompt="""You are a helpful customer support agent with access to customer history and order information. 
    
    Your role:
    - Help customers with their orders, returns, and product issues
    - Use customer context to provide personalized support
    - Search for product information when needed
    - Be empathetic and solution-focused
    - Reference previous orders and preferences when relevant
    
    Always be professional, helpful, and aim to resolve customer issues efficiently."""
)

print("MemorySession 통합을 갖춘 고객 지원 에이전트가 생성되었습니다")

### 7단계: 고객 이력 시드 데이터 추가

메모리 기능을 시연하기 위해 이전 고객 상호작용을 추가합니다.

**참고: 이 섹션에서는 ConversationalMessage 형식과 세션 기반 저장을 사용합니다.**

In [ ]:
# MemorySession을 사용하여 이전 고객 상호작용 시드 데이터 추가
previous_interactions = [
    ConversationalMessage("I bought a new iPhone 15 Pro on June 1st, 2025. Order number is 123456.", USER),
    ConversationalMessage("Thank you for your purchase! I can see your iPhone 15 Pro order #123456 was delivered successfully. How can I help you today?", ASSISTANT),
    ConversationalMessage("I also ordered Sennheiser headphones on June 20th. Order number 654321. They came with 1-year warranty.", USER),
    ConversationalMessage("Perfect! I have your Sennheiser headphones order #654321 on file with the 1-year warranty. Both your iPhone and headphones should work great together.", ASSISTANT),
    ConversationalMessage("I'm looking for a good laptop. I prefer ThinkPad models.", USER),
    ConversationalMessage("Great choice! ThinkPads are excellent for their durability and performance. Let me help you find the right model for your needs.", ASSISTANT)
]

# MemorySession을 사용하여 저장
try:
    event_response = customer_session.add_turns(previous_interactions)
    logger.info(f"MemorySession을 사용하여 고객 이력 시드 데이터 추가 완료")
    logger.info(f"   이벤트 ID: {event_response['eventId']}")
except Exception as e:
    logger.error(f"이력 시드 데이터 추가 오류: {e}")

#### 에이전트가 준비되었습니다.

### 고객 지원 시나리오를 테스트해 봅시다

In [ ]:
# 테스트 1: 고객이 iPhone 문제를 보고합니다
logger.info("테스트 1 실행: iPhone 성능 문제")
# iPhone이 매우 느리고 충전 시 뜨거워지는 문제
test_query_1 = "My iPhone is running very slow and gets hot when charging. Can you help?"
logger.info(f"쿼리: {test_query_1}")

response1 = support_agent(test_query_1)
logger.info(f"테스트 1이 성공적으로 완료되었습니다")
print(f"\niPhone 문제 지원 응답:\n{response1}\n")

In [ ]:
# 테스트 2: 블루투스 연결 문제
logger.info("테스트 2 실행: 블루투스 연결 문제")
# iPhone이 Sennheiser 헤드폰과 블루투스로 연결되지 않는 문제
test_query_2 = "My iPhone won't connect to my Sennheiser headphones via Bluetooth. How do I fix this?"
logger.info(f"쿼리: {test_query_2}")

response2 = support_agent(test_query_2)
logger.info(f"테스트 2가 성공적으로 완료되었습니다")
print(f"\n블루투스 문제 지원 응답:\n{response2}\n")

In [ ]:
# 테스트 3: 주문 상태 확인
logger.info("테스트 3 실행: 주문 상태 확인")
# 최근 주문 상태를 확인해 달라는 요청
test_query_3 = "Can you check the status of my recent orders?"
logger.info(f"쿼리: {test_query_3}")

response3 = support_agent(test_query_3)
logger.info(f"테스트 3이 성공적으로 완료되었습니다")
print(f"\n주문 상태 지원 응답:\n{response3}\n")

In [ ]:
# 테스트 4: 선호도 기반 제품 추천
logger.info("테스트 4 실행: 제품 추천")
# 노트북 구매에 관심이 있으며 ThinkPad 모델 추천 요청
test_query_4 = "I'm still interested in buying a laptop. What ThinkPad models do you recommend?"
logger.info(f"쿼리: {test_query_4}")

response4 = support_agent(test_query_4)
logger.info(f"테스트 4가 성공적으로 완료되었습니다")
print(f"\n제품 추천 지원 응답:\n{response4}\n")

logger.info("모든 고객 지원 시나리오 테스트가 완료되었습니다!")

## 고급 기능: 분기 및 메타데이터

### SessionManager를 사용한 대화 분기

분기를 사용하여 대안적인 지원 시나리오를 탐색합니다:

In [ ]:
# 대화에서 마지막 이벤트 ID를 가져옵니다
events = customer_session.list_events()
if events:
    last_event_id = events[-1].eventId
    
    # 프리미엄 지원 경로를 탐색하기 위해 대화를 분기합니다
    branch_event = customer_session.fork_conversation(
        root_event_id=last_event_id,
        branch_name="premium-support",
        messages=[
            ConversationalMessage(
                "I'd like to upgrade to premium support for faster resolution.",
                USER
            ),
            ConversationalMessage(
                "Excellent choice! With premium support, you'll get 24/7 priority assistance, dedicated account manager, and same-day resolution guarantee. Let me process your upgrade.",
                ASSISTANT
            )
        ]
    )
    
    logger.info(f"이벤트 {last_event_id}에서 프리미엄 지원 분기가 생성되었습니다")
    
    # 모든 분기 목록 조회
    branches = customer_session.list_branches()
    print(f"\n지원 세션에 {len(branches)}개의 분기가 있습니다:")
    for branch in branches:
        print(f"   - {branch.name}: {branch.event_count}개의 이벤트")
else:
    print("분기할 이벤트를 찾을 수 없습니다")

### 고급 지원 추적을 위한 메타데이터

메타데이터를 사용하여 포괄적인 지원 메트릭을 추적합니다:

In [ ]:
from bedrock_agentcore.memory.models import StringValue

# 포괄적인 메타데이터와 함께 지원 상호작용 추가
metadata_event = customer_session.add_turns(
    messages=[
        ConversationalMessage(
            "The ThinkPad X1 Carbon you recommended is perfect! I'll order it now.",
            USER
        ),
        ConversationalMessage(
            "Fantastic! The ThinkPad X1 Carbon is an excellent choice for your needs. I'll help you complete the order with your preferred configuration.",
            ASSISTANT
        )
    ],
    metadata={
        "interaction_type": StringValue.build("product_recommendation"),
        "outcome": StringValue.build("purchase_intent"),
        "product_category": StringValue.build("laptops"),
        "product_brand": StringValue.build("lenovo"),
        "customer_sentiment": StringValue.build("positive"),
        "support_tier": StringValue.build("standard"),
        "session_duration_minutes": StringValue.build("15")
    }
)

logger.info(f"메타데이터가 포함된 지원 이벤트 추가 완료 - 이벤트 ID: {metadata_event['eventId']}")
print("\n지원 상호작용에 태그가 지정되었습니다:")
print("   - 상호작용 유형: product_recommendation")
print("   - 결과: purchase_intent")
print("   - 제품 카테고리: laptops")
print("   - 고객 감정: positive")

### 고급 메타데이터 쿼리

지원 패턴과 고객 행동을 분석합니다:

In [ ]:
from bedrock_agentcore.memory.models import EventMetadataFilter, LeftExpression, RightExpression, OperatorType

try:
    # 제품 추천 상호작용 쿼리
    recommendation_events = customer_session.list_events(
        eventMetadata=[
            {
                "left": {"metadataKey": "interaction_type"},
                "operator": "EQUALS_TO",
                "right": {"metadataValue": {"stringValue": "product_recommendation"}}
            }
        ]
    )
    
    print(f"\n{len(recommendation_events)}개의 제품 추천 상호작용을 찾았습니다")
    
    # 긍정적 감정 상호작용 쿼리
    positive_events = customer_session.list_events(
        eventMetadata=[
            {
                "left": {"metadataKey": "customer_sentiment"},
                "operator": "EQUALS_TO",
                "right": {"metadataValue": {"stringValue": "positive"}}
            }
        ]
    )
    
    print(f"{len(positive_events)}개의 긍정적 감정 상호작용을 찾았습니다")
    
    print("\n고급 분석 사용 사례:")
    print("   - 추천에서 구매까지의 전환율 추적")
    print("   - 시간에 따른 고객 감정 트렌드 분석")
    print("   - 가장 효과적인 지원 상호작용 유형 식별")
    print("   - 지원 등급 성능 측정")
    print("   - 상세한 고객 여정 보고서 생성")
    print("   - 제품 추천 전략 최적화")
    
except Exception as e:
    logger.error(f"메타데이터 쿼리 오류: {e}")
    print(f"참고: 메타데이터 필터링에는 메타데이터 태그가 있는 이벤트가 필요합니다")

#### 고객 지원 튜토리얼이 완료되었습니다!
주요 내용:
- 메모리 훅은 MemorySessionManager를 사용하여 지원 세션 간 고객 컨텍스트를 자동으로 관리합니다
- 타입화된 전략 클래스를 사용하는 다중 전략 메모리가 대화에서 주문, 선호도, 사실을 캡처합니다
- 에이전트는 고객 이력을 기반으로 개인화된 지원을 제공할 수 있습니다
- 주문 조회 및 웹 검색 기능을 위한 도구가 통합될 수 있습니다
- 영구 메모리로 고객 지원이 더 효율적이 됩니다
- **분기를 통해 대안적인 지원 접근 방식 및 에스컬레이션 경로를 테스트할 수 있습니다**
- **메타데이터는 포괄적인 지원 분석 및 고객 여정 추적을 제공합니다**

## 정리

### 선택사항: 메모리 리소스 삭제

In [ ]:
# MemoryManager를 사용하여 메모리 리소스를 삭제하려면 주석을 해제하세요
# try:
#     memory_manager.delete_memory(memory_id)
#     print(f"MemoryManager를 사용하여 메모리 리소스 삭제 완료: {memory_id}")
# except Exception as e:
#     print(f"메모리 삭제 오류: {e}")